In [6]:
%%writefile matmul.cu
#include <iostream>
#include <cuda_runtime.h>
#include <limits>   // for input validation

using namespace std;

// CUDA Kernel
__global__ void multiply(int* A, int* B, int* C,
                         int rowsA, int colsA, int colsB) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rowsA && col < colsB) {
        int sum = 0;
        for (int i = 0; i < colsA; i++) {
            sum += A[row * colsA + i] * B[i * colsB + col];
        }
        C[row * colsB + col] = sum;
    }
}

// Print matrix
void print(int* matrix, int rows, int cols) {
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            cout << matrix[i * cols + j] << " ";
        }
        cout << "\n";
    }
    cout << "\n";
}

// 🔹 Safe integer input
int getInt(string message) {
    int value;
    while (true) {
        cout << message;
        cin >> value;

        if (cin.fail()) {
            cout << "Invalid input! Enter a number.\n";
            cin.clear();
            cin.ignore(numeric_limits<streamsize>::max(), '\n');
        } else if (value <= 0) {
            cout << "Value must be > 0.\n";
        } else {
            return value;
        }
    }
}

// 🔹 Safe matrix input
void inputMatrix(int* mat, int rows, int cols, string name) {
    cout << "Enter elements of " << name << " (" << rows << " x " << cols << "):\n";

    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            while (true) {
                cout << name << "[" << i << "][" << j << "] = ";
                cin >> mat[i * cols + j];

                if (cin.fail()) {
                    cout << "Invalid input! Enter a number.\n";
                    cin.clear();
                    cin.ignore(numeric_limits<streamsize>::max(), '\n');
                } else {
                    break;
                }
            }
        }
    }
}

int main() {
    int rowsA, colsA, rowsB, colsB;

    // 🔹 Input with validation
    rowsA = getInt("Enter rows of Matrix A: ");
    colsA = getInt("Enter cols of Matrix A: ");

    while (true) {
        rowsB = getInt("Enter rows of Matrix B: ");
        colsB = getInt("Enter cols of Matrix B: ");

        if (colsA != rowsB) {
            cout << "❌ Invalid! cols(A) must equal rows(B). Try again.\n";
        } else {
            break;
        }
    }

    // Allocate
    int *A = new int[rowsA * colsA];
    int *B = new int[rowsB * colsB];
    int *C = new int[rowsA * colsB];

    // Input matrices
    inputMatrix(A, rowsA, colsA, "A");
    inputMatrix(B, rowsB, colsB, "B");

    cout << "\nMatrix A:\n";
    print(A, rowsA, colsA);

    cout << "Matrix B:\n";
    print(B, rowsB, colsB);

    // Device memory
    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, rowsA * colsA * sizeof(int));
    cudaMalloc(&d_B, rowsB * colsB * sizeof(int));
    cudaMalloc(&d_C, rowsA * colsB * sizeof(int));

    cudaMemcpy(d_A, A, rowsA * colsA * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, rowsB * colsB * sizeof(int), cudaMemcpyHostToDevice);

    // Threads & Blocks
    int THREADS = 16;
    dim3 threads(THREADS, THREADS);
    dim3 blocks((colsB + THREADS - 1) / THREADS,
                (rowsA + THREADS - 1) / THREADS);

    multiply<<<blocks, threads>>>(d_A, d_B, d_C,
                                 rowsA, colsA, colsB);

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, rowsA * colsB * sizeof(int),
               cudaMemcpyDeviceToHost);

    cout << "Result Matrix:\n";
    print(C, rowsA, colsB);

    // Cleanup
    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


Overwriting matmul.cu


In [ ]:
!nvidia-smi

Wed May 13 10:02:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [8]:
!nvcc matmul.cu -o matmul && ./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter rows of Matrix A: 2 2
Enter cols of Matrix A: Enter rows of Matrix B: 2 2
Enter cols of Matrix B: Enter elements of A (2 x 2):
A[0][0] = 1
A[0][1] = 3
A[1][0] = 4
A[1][1] = 5
Enter elements of B (2 x 2):
B[0][0] = 3
B[0][1] = 5
B[1][0] = 6
B[1][1] = 7

Matrix A:
1 3 
4 5 

Matrix B:
3 5 
6 7 

Result Matrix:
21 26 
42 55 

